# prep_check — verification + pipeline prototype (scratch; does NOT train the final model)
Verifies the EDA claims against `orders-sheet.csv`, prototypes the preprocessing pipeline and runs sanity checks.
`eda.ipynb` is untouched. Supports `REPORT_preprocessing.md`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib, time, io
from scipy.stats import spearmanr, chi2_contingency
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, average_precision_score
pd.options.display.width = 200; pd.options.display.max_columns = 40
raw = pd.read_csv("orders-sheet.csv")
raw.insert(0, "row_id", np.arange(len(raw)))          # safe row identifier (order_id is NOT unique)
print(raw.shape)

(150000, 23)


## 1. Verify the EDA summary claims

In [2]:
print("missing %:\n", raw.isna().mean().mul(100).round(2)[lambda s: s > 0].to_dict())
print("exact duplicate rows (keep first):", raw.drop(columns="row_id").duplicated().sum(),
      "| rows involved:", raw.drop(columns="row_id").duplicated(keep=False).sum())
print("unique order_id:", raw.order_id.nunique(), "| rows whose order_id repeats:", raw.order_id.duplicated(keep=False).sum())
n = raw.order_id.str[3:].astype(int)
print("order_id numeric range:", n.min(), "-", n.max(), "| expected #unique if 150k IDs were drawn at random from that range:",
      round(n.max() * (1 - np.exp(-len(raw) / n.max()))))
same = raw[~raw.drop(columns="row_id").duplicated(keep=False) & raw.order_id.duplicated(keep=False)]
print("repeated-ID rows that are NOT exact duplicates:", len(same), "| same (order_id,customer_id) but different row:",
      raw.duplicated(["order_id", "customer_id"], keep=False).sum() - raw.drop(columns="row_id").duplicated(keep=False).sum())

missing %:
 {'customer_age': 1.2, 'discount_pct': 0.6, 'shipping_cost': 0.4, 'device_type': 1.8, 'ip_country': 0.9}


exact duplicate rows (keep first): 350 | rows involved: 700
unique order_id: 74002 | rows whose order_id repeats: 120477
order_id numeric range: 2 - 92000 | expected #unique if 150k IDs were drawn at random from that range: 73982


repeated-ID rows that are NOT exact duplicates: 119777 | same (order_id,customer_id) but different row: 14


## 2. Cleaning — label canonicalisation (fixed mapping) + pipeline classes

In [3]:
CAT_COLS = ["customer_country", "ip_country", "category", "payment_method", "sales_channel", "device_type", "order_status", "coupon_used"]
FIXED_MAP = {"payment_method": {"paypal": "PayPal"}, "sales_channel": {"web": "Web"}}

def clean_labels(df):
    df = df.copy()
    for c in CAT_COLS:
        v = df[c].astype("string").str.strip()
        if c == "category":
            v = v.str.title()
        elif c in FIXED_MAP:
            v = v.str.lower().map(FIXED_MAP[c]).fillna(v)
        df[c] = v
    return df

df = clean_labels(raw)
print({c: (raw[c].nunique(), df[c].nunique()) for c in CAT_COLS}, "  (raw_unique, clean_unique)")
print({c: sorted(df[c].dropna().unique()) for c in ["payment_method", "sales_channel"]})
dup = raw.drop(columns="row_id").duplicated(keep="first").values

{'customer_country': (10, 10), 'ip_country': (10, 10), 'category': (20, 10), 'payment_method': (6, 5), 'sales_channel': (4, 3), 'device_type': (4, 4), 'order_status': (5, 5), 'coupon_used': (2, 2)}   (raw_unique, clean_unique)
{'payment_method': ['Apple Pay', 'Bank Transfer', 'Card', 'Google Pay', 'PayPal'], 'sales_channel': ['Marketplace', 'Mobile App', 'Web']}


### Formula check (denominators matter)

In [4]:
sub = df.quantity * df.unit_price * (1 - df.discount_pct / 100)
exp_ = sub + df.shipping_cost + df.tax_amount
res = df.order_value - exp_
ev = res.notna(); viol = (res.abs() > 0.01) & ev
print(f"evaluable rows {ev.sum()} | match {(~viol & ev).sum()} = {(~viol & ev).sum()/ev.sum()*100:.2f}% of evaluable, "
      f"{(~viol & ev).sum()/len(df)*100:.2f}% of all rows | violations {viol.sum()} ({viol.mean()*100:.2f}% of all) | NaN residual {(~ev).sum()}")
print("violations: share with order_value > expected:", round((res[viol] > 0).mean(), 3), "| median |residual|:", round(res[viol].abs().median(), 1),
      "| min |residual|:", round(res[viol].abs().min(), 2))
print("rows with 0.01 < |res| <= 1:", ((res.abs() > 0.01) & (res.abs() <= 1)).sum(), "  (tolerance 0.01 vs 1.0 changes the count by 1 -> violations are not rounding noise)")
print("residual NaN because: discount missing", df.discount_pct.isna().sum(), "| shipping missing", df.shipping_cost.isna().sum(),
      "| both", (df.discount_pct.isna() & df.shipping_cost.isna()).sum())
print("tax_rate on formula-OK rows takes values (share):", (df.tax_amount / sub)[~viol & ev].round(2).value_counts(normalize=True).head(3).round(3).to_dict(), "... -> ~uniform over 0.00-0.20 in 1% steps")
print("fee_rate (fee/subtotal) median by channel:", (df.platform_fee / sub).groupby(df.sales_channel).median().round(4).to_dict())

evaluable rows 148502 | match 146360 = 98.56% of evaluable, 97.57% of all rows | violations 2142 (1.43% of all) | NaN residual 1498
violations: share with order_value > expected: 0.701 | median |residual|: 2563.2 | min |residual|: 0.96
rows with 0.01 < |res| <= 1: 1   (tolerance 0.01 vs 1.0 changes the count by 1 -> violations are not rounding noise)
residual NaN because: discount missing 900 | shipping missing 601 | both 3
tax_rate on formula-OK rows takes values (share): {0.1: 0.051, 0.03: 0.051, 0.02: 0.05} ... -> ~uniform over 0.00-0.20 in 1% steps
fee_rate (fee/subtotal) median by channel: {'Marketplace': 0.1153, 'Mobile App': 0.1152, 'Web': 0.1153}


### Rule-derived PSEUDO-labels (for sanity checks only — NOT ground truth, and partly circular with the features)
Reverse-engineered from the data: five tight clusters of rows that break a domain rule by a wide margin.

In [5]:
pm = df.groupby("product_id").unit_price.transform("median")
fee_rate = df.platform_fee / sub
pr = df.unit_price / pm
typ = np.full(len(df), "clean", dtype=object)
typ[(df.quantity > 20).values] = "A_bulk_highprice"            # qty 25-99, price 2-620x product median
typ[(df.discount_pct > 50).values] = "B_extreme_discount"      # discount 70-95%
typ[((df.shipping_cost > 100) | (pr < 0.5)).values] = "C_ship_or_lowprice"   # shipping 153-899 / price << median
typ[(viol & (typ == "clean")).values] = "E_newacct_value_inflated"           # inputs normal, order_value x2-640, acct age 0-7d
typ[((fee_rate > 0.3) & (typ == "clean")).values] = "F_fee_inflated"          # fee 0.39-3.6x subtotal
y = typ != "clean"
print(pd.Series(typ).value_counts().to_dict(), "| total", y.sum(), f"= {y.mean()*100:.2f}%")
print("legit fee_rate range (non-F):", fee_rate[typ != "F_fee_inflated"].quantile([0, .999]).round(3).to_dict(), "(A/B/C rows included, they distort sub) ; clean-only:", fee_rate[typ == "clean"].agg(["min", "max"]).round(3).to_dict())
print("residual violators cover", round((viol & y).sum() / y.sum() * 100, 1), "% of pseudo-anomalies; fee-inflated rows with formula violation:", viol[typ == "F_fee_inflated"].sum())
print("E type account_age_days:", df.account_age_days[typ == "E_newacct_value_inflated"].describe()[["min", "50%", "max"]].to_dict(),
      "| geo mismatch share:", (df.customer_country != df.ip_country)[typ == "E_newacct_value_inflated"].mean())
print("anomaly rate for account_age<=30d:", round(y[df.account_age_days <= 30].mean() * 100, 1), "% vs", round(y[df.account_age_days > 30].mean() * 100, 2), "% otherwise")
print("median order_value account<=30d (all rows):", df.order_value[df.account_age_days <= 30].median(), "| overall:", df.order_value.median(), "| clean-only <=30d:", df.order_value[(df.account_age_days <= 30) & ~y].median())

{'clean': 147305, 'B_extreme_discount': 543, 'C_ship_or_lowprice': 540, 'A_bulk_highprice': 539, 'F_fee_inflated': 537, 'E_newacct_value_inflated': 536} | total 2695 = 1.80%
legit fee_rate range (non-F): {0.0: 0.0, 0.999: 1.941} (A/B/C rows included, they distort sub) ; clean-only: {'min': 0.069, 'max': 0.17}
residual violators cover 79.5 % of pseudo-anomalies; fee-inflated rows with formula violation: 0
E type account_age_days: {'min': 0.0, '50%': 3.0, 'max': 7.0} | geo mismatch share: 1.0
anomaly rate for account_age<=30d: 26.8 % vs 1.44 % otherwise
median order_value account<=30d (all rows): 81.32 | overall: 51.45 | clean-only <=30d: 52.13


### Do other columns / customer-level structure carry signal?

In [6]:
for c in ["payment_method", "sales_channel", "device_type", "order_status", "coupon_used", "category", "customer_country"]:
    print(f"{c:17s} chi2 p={chi2_contingency(pd.crosstab(df[c].fillna('NA'), y))[1]:.3f}", end=" | ")
ts = pd.to_datetime(df.order_timestamp)
print("\nhour p=%.3f dow p=%.3f" % (chi2_contingency(pd.crosstab(ts.dt.hour, y))[1], chi2_contingency(pd.crosstab(ts.dt.dayofweek, y))[1]),
      "| customer_age mean anomalous/normal:", round(df.customer_age[y].mean(), 1), round(df.customer_age[~y].mean(), 1))
geo = (df.customer_country != df.ip_country)[df.ip_country.notna()]
print("geo mismatch share", round(geo.mean(), 3), "| anomaly rate mismatch vs match:", round(y[geo.index][geo].mean() * 100, 2), round(y[geo.index][~geo].mean() * 100, 2))
from scipy.stats import binom
cs = pd.DataFrame({"y": y, "c": df.customer_id.values}).groupby("c").y.agg(["sum", "size"])
print("customers with >=2 pseudo-anomalies:", (cs["sum"] >= 2).sum(), "| expected if independent:", round(sum(binom.sf(1, k, y.mean()) for k in cs["size"]), 1),
      "| customers:", len(cs), "| orders/customer mean", round(cs["size"].mean(), 1))
for c in ["customer_age", "discount_pct", "shipping_cost", "device_type", "ip_country"]:
    m = df[c].isna(); print(f"{c} missing n={m.sum()} anomaly rate missing/non-missing {y[m].mean()*100:.2f}% / {y[~m].mean()*100:.2f}%", end=" | ")
print()
combo = df.groupby(["sales_channel", "payment_method", "customer_country"]).order_id.transform("size")
print("3-way combos:", combo.groupby([df.sales_channel, df.payment_method, df.customer_country]).first().shape[0], "| smallest combo:", combo.min(),
      "| anomaly rate in rarest decile vs rest:", round(y[combo <= combo.quantile(.1)].mean() * 100, 2), round(y[combo > combo.quantile(.1)].mean() * 100, 2))
print("per-product price: log(price/product median) clean quantiles:", np.log(pr[~y]).quantile([.001, .5, .999]).round(2).to_dict(), "| products:", df.product_id.nunique(),
      "| rows/product min", df.groupby("product_id").size().min())
print("category median unit_price:", df.groupby("category").unit_price.median().round(1).agg(["min", "max"]).to_dict(), "| category median order_value:", df.groupby("category").order_value.median().round(1).agg(["min", "max"]).to_dict())

payment_method    chi2 p=0.505 | sales_channel     chi2 p=0.468 | device_type       chi2 p=0.361 | order_status      chi2 p=0.294 | coupon_used       chi2 p=0.259 | 

category          chi2 p=0.989 | customer_country  chi2 p=0.050 | 


hour p=0.903 dow p=0.670 | customer_age mean anomalous/normal: 46.8 46.6
geo mismatch share 0.811 | anomaly rate mismatch vs match: 1.89 1.39


customers with >=2 pseudo-anomalies: 193 | expected if independent: 182.8 | customers: 17997 | orders/customer mean 8.3
customer_age missing n=1802 anomaly rate missing/non-missing 1.94% / 1.79% | discount_pct missing n=900 anomaly rate missing/non-missing 1.22% / 1.80% | shipping_cost missing n=601 anomaly rate missing/non-missing 1.33% / 1.80% | device_type missing n=2703 anomaly rate missing/non-missing 1.48% / 1.80% | ip_country missing n=1347 anomaly rate missing/non-missing 1.86% / 1.80% | 
3-way combos: 150 | smallest combo: 63 | anomaly rate in rarest decile vs rest: 2.01 1.77
per-product price: log(price/product median) clean quantiles: {0.001: -0.24, 0.5: 0.0, 0.999: 0.21} | products: 1200 | rows/product min 87
category median unit_price: {'min': 24.9, 'max': 34.2} | category median order_value: {'min': 45.9, 'max': 54.1}


## 3. Skew before/after log (transform decisions)

In [7]:
num = ["quantity", "unit_price", "discount_pct", "shipping_cost", "tax_amount", "platform_fee", "order_value", "customer_age", "account_age_days"]
print(pd.DataFrame({"skew_raw": df[num].skew(), "skew_log1p": np.log1p(df[num]).skew()}).round(2).T)

            quantity  unit_price  discount_pct  shipping_cost  tax_amount  platform_fee  order_value  customer_age  account_age_days
skew_raw       14.72       11.21          1.75          19.52        9.04          7.52        23.17         -0.01             -0.00
skew_log1p      2.08        0.23          0.26          -0.27        0.67          0.68         1.49         -0.50             -2.13


## 4. The pipeline: custom transformer (fit = learn lookups on TRAIN only, transform = stateless row-wise)

In [8]:
FULL = ["log_quantity", "log_unit_price", "discount_pct", "log_shipping", "log1p_tax_rate", "log_fee_rate", "price_vs_product",
        "log_order_value", "log_value_ratio", "log_acct_age", "log_ship_ratio", "log_fee_to_value"]
NO_VALUE = ["log_quantity", "log_unit_price", "discount_pct", "log_shipping", "log1p_tax_rate", "log_fee_rate", "price_vs_product", "log_acct_age", "log_ship_ratio"]

class OrderFeatures(BaseEstimator, TransformerMixin):
    """Raw order rows -> engineered numeric features. All lookups are learned in fit() from training rows only."""
    def __init__(self, features=tuple(FULL)):
        self.features = features
    def fit(self, df, y=None):
        d = clean_labels(df)
        self.prod_med_price_ = d.groupby("product_id").unit_price.median().to_dict()
        self.cat_med_price_ = d.groupby("category").unit_price.median().to_dict()
        self.global_med_price_ = float(d.unit_price.median())
        self.ship_median_ = float(d.shipping_cost.median())
        return self
    def transform(self, df):
        d = clean_labels(df)
        disc = d.discount_pct.fillna(0.0); ship = d.shipping_cost.fillna(self.ship_median_)
        res_ok = d.discount_pct.notna() & d.shipping_cost.notna()          # residual only trusted when nothing was imputed
        q, p = d.quantity, d.unit_price
        sub = (q * p * (1 - disc / 100)).clip(lower=0.01)
        expected = sub + ship + d.tax_amount
        pm = d.product_id.map(self.prod_med_price_).fillna(d.category.map(self.cat_med_price_)).fillna(self.global_med_price_)
        X = pd.DataFrame(index=d.index)
        X["log_quantity"] = np.log1p(q)
        X["log_unit_price"] = np.log(p)
        X["discount_pct"] = disc
        X["log_shipping"] = np.log1p(ship)
        X["log1p_tax_rate"] = np.log1p(d.tax_amount / sub)
        X["log_fee_rate"] = np.log((d.platform_fee / sub).clip(lower=1e-4))
        X["log_ship_ratio"] = np.log1p(ship / sub)
        X["price_vs_product"] = np.log(p / pm)
        X["log_order_value"] = np.log(d.order_value)
        X["log_fee_to_value"] = np.log((d.platform_fee / d.order_value).clip(lower=1e-6))
        X["log_value_ratio"] = np.where(res_ok, np.log(d.order_value / expected), 0.0)
        X["log_acct_age"] = np.log1p(d.account_age_days)
        return X[list(self.features)]

train_raw = raw[~dup]
fe = OrderFeatures().fit(train_raw)
X = fe.transform(raw)
print("NaNs in feature matrix:", int(X.isna().sum().sum()), "| shape", X.shape)
print(X.describe(percentiles=[.001, .5, .999]).T[["min", "0.1%", "50%", "99.9%", "max"]].round(2))

NaNs in feature matrix: 0 | shape (150000, 12)
                    min   0.1%   50%  99.9%    max
log_quantity       0.69   0.69  0.69   4.39   4.61
log_unit_price     0.00   0.76  3.39   7.26   7.50
discount_pct       0.00   0.00  0.00  89.00  95.00
log_shipping       0.00   0.00  2.02   6.58   6.80
log1p_tax_rate     0.00   0.00  0.10   1.00   3.72
log_fee_rate      -9.21  -9.21 -2.16   0.69   3.56
price_vs_product  -5.53  -2.70  0.00   4.04   6.43
log_order_value    0.21   1.37  3.94  11.38  12.21
log_value_ratio   -4.89  -2.79  0.00   5.58   7.97
log_acct_age       0.00   0.69  7.31   8.01   8.01
log_ship_ratio     0.00   0.00  0.13   4.96   6.99
log_fee_to_value -13.52 -10.22 -2.45  -0.34  -0.16


### Correlation / redundancy of the candidate set (incl. features we propose to DROP)

In [9]:
ALL = FULL + ["res_slog", "abs_res_log", "log_value_vs_cat", "geo_mismatch", "customer_age"]
d_ = clean_labels(raw)
extra = pd.DataFrame(index=raw.index)
sub_c = sub.clip(lower=0.01)
rr = (df.order_value - (sub_c + df.shipping_cost.fillna(df.shipping_cost.median()) + df.tax_amount))
okm = df.discount_pct.notna() & df.shipping_cost.notna()
rr = rr.where(okm, 0.0)
extra["res_slog"] = np.sign(rr) * np.log1p(rr.abs()); extra["abs_res_log"] = np.log1p(rr.abs())
extra["log_value_vs_cat"] = np.log(df.order_value / df.groupby("category").order_value.transform("median"))
extra["geo_mismatch"] = (df.customer_country != df.ip_country).astype(float).where(df.ip_country.notna(), 0.0)
extra["customer_age"] = df.customer_age.fillna(df.customer_age.median())
XA = pd.concat([X, extra], axis=1)[ALL]
C = XA.corr(method="spearman")
print("|Spearman| >= 0.7:", sorted([(a, b, round(C.loc[a, b], 2)) for i, a in enumerate(ALL) for b in ALL[i + 1:] if abs(C.loc[a, b]) >= 0.7], key=lambda t: -abs(t[2])))
from sklearn.linear_model import LinearRegression
print("R2 of each FULL feature explained by the other 11 (linear):",
      {f: round(LinearRegression().fit(X.drop(columns=f), X[f]).score(X.drop(columns=f), X[f]), 2) for f in FULL})
print("formula_ok share:", round((rr.abs() <= 0.01).mean(), 4), "| res_slog exactly 0 share:", round((extra.res_slog == 0).mean(), 3))

|Spearman| >= 0.7: [('log_order_value', 'log_value_vs_cat', np.float64(1.0)), ('log_value_ratio', 'res_slog', np.float64(0.99)), ('log_unit_price', 'log_order_value', np.float64(0.79)), ('log_unit_price', 'log_value_vs_cat', np.float64(0.78)), ('log_fee_rate', 'log_fee_to_value', np.float64(0.74))]


R2 of each FULL feature explained by the other 11 (linear): {'log_quantity': 0.99, 'log_unit_price': 0.99, 'discount_pct': 0.83, 'log_shipping': 0.39, 'log1p_tax_rate': 0.53, 'log_fee_rate': 0.99, 'price_vs_product': 0.69, 'log_order_value': 0.99, 'log_value_ratio': 0.98, 'log_acct_age': 0.05, 'log_ship_ratio': 0.97, 'log_fee_to_value': 0.99}
formula_ok share: 0.9857 | res_slog exactly 0 share: 0.401


## 5. Modeling-readiness experiments (feature ablation vs pseudo-labels; remember the circularity caveat)

In [10]:
K = int(y.sum())
def fit_if(cols, seed=0, n=300, ms=256, Xd=None):
    Xd = X[cols] if Xd is None else Xd
    m = IsolationForest(n_estimators=n, max_samples=ms, random_state=seed, n_jobs=-1).fit(Xd[~dup])
    return m, -m.score_samples(Xd)
TYPES = ["A_bulk_highprice", "B_extreme_discount", "C_ship_or_lowprice", "E_newacct_value_inflated", "F_fee_inflated"]
def rep(name, s, k=K):
    top = np.zeros(len(s), bool); top[np.argsort(-s)[:k]] = True
    print(f"{name:46s} AUC={roc_auc_score(y, s):.3f} AP={average_precision_score(y, s):.3f} P@K={top[y].mean():.3f}  recall@K:", {t[0]: round(top[typ == t].mean(), 2) for t in TYPES})
Xraw = raw[num].fillna(raw[num].median())
rep("V0 raw numerics, nothing engineered", fit_if(None, Xd=Xraw)[1])
rep("V1 log1p of raw numerics", fit_if(None, Xd=np.log1p(Xraw))[1])
G1 = ["log_quantity", "log_unit_price", "discount_pct", "log_shipping", "log1p_tax_rate", "log_fee_rate", "price_vs_product"]
rep("V2 stated inputs only (7), no acct age", fit_if(G1)[1])
rep("V3 NO_VALUE (9: V2 + log_acct_age + ship_ratio)", fit_if(NO_VALUE)[1])
rep("V4 V2 + log_order_value", fit_if(G1 + ["log_order_value"])[1])
rep("V5 V2 + order_value + value_ratio", fit_if(G1 + ["log_order_value", "log_value_ratio"])[1])
rep("V6 V5 + log_acct_age", fit_if(G1 + ["log_order_value", "log_value_ratio", "log_acct_age"])[1])
rep("V6r V5 + RAW account_age_days", fit_if(None, Xd=pd.concat([X[G1 + ["log_order_value", "log_value_ratio"]], raw.account_age_days], axis=1))[1])
rep("V6b V5 + acct_new7 flag", fit_if(None, Xd=pd.concat([X[G1 + ["log_order_value", "log_value_ratio"]], (raw.account_age_days <= 7).astype(int).rename("new7")], axis=1))[1])
rep("FULL (12) = V6 + ship_ratio + fee_to_value", fit_if(FULL)[1])
rep("FULL + geo_mismatch", fit_if(None, Xd=pd.concat([X, extra.geo_mismatch], axis=1))[1])
rep("FULL + customer_age", fit_if(None, Xd=pd.concat([X, extra.customer_age], axis=1))[1])
rep("FULL + hour", fit_if(None, Xd=pd.concat([X, ts.dt.hour.rename("hour")], axis=1))[1])
rep("FULL + res_slog (collinear w/ value_ratio)", fit_if(None, Xd=pd.concat([X, extra.res_slog], axis=1))[1])
rep("FULL + log_value_vs_cat (collinear w/ order_value)", fit_if(None, Xd=pd.concat([X, extra.log_value_vs_cat], axis=1))[1])
rep("FULL minus log_fee_rate (drop fee signal)", fit_if([c for c in FULL if c != "log_fee_rate" and c != "log_fee_to_value"])[1])

V0 raw numerics, nothing engineered            AUC=0.926 AP=0.251 P@K=0.252  recall@K: {'A': np.float64(1.0), 'B': np.float64(0.12), 'C': np.float64(0.06), 'E': np.float64(0.04), 'F': np.float64(0.05)}


V1 log1p of raw numerics                       AUC=0.923 AP=0.366 P@K=0.408  recall@K: {'A': np.float64(1.0), 'B': np.float64(0.1), 'C': np.float64(0.06), 'E': np.float64(0.86), 'F': np.float64(0.02)}


V2 stated inputs only (7), no acct age         AUC=0.886 AP=0.649 P@K=0.614  recall@K: {'A': np.float64(1.0), 'B': np.float64(0.99), 'C': np.float64(0.95), 'E': np.float64(0.01), 'F': np.float64(0.11)}


V3 NO_VALUE (9: V2 + log_acct_age + ship_ratio) AUC=0.978 AP=0.744 P@K=0.664  recall@K: {'A': np.float64(1.0), 'B': np.float64(0.98), 'C': np.float64(1.0), 'E': np.float64(0.22), 'F': np.float64(0.11)}


V4 V2 + log_order_value                        AUC=0.964 AP=0.696 P@K=0.630  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(0.95), 'E': np.float64(0.11), 'F': np.float64(0.08)}


...

FULL + customer_age                            AUC=0.982 AP=0.852 P@K=0.810  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.05)}


FULL + hour                                    AUC=0.979 AP=0.847 P@K=0.806  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.03)}


FULL + res_slog (collinear w/ value_ratio)     AUC=0.985 AP=0.856 P@K=0.811  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.05)}


FULL + log_value_vs_cat (collinear w/ order_value) AUC=0.982 AP=0.850 P@K=0.806  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(0.99), 'E': np.float64(1.0), 'F': np.float64(0.04)}


FULL minus log_fee_rate (drop fee signal)      AUC=0.902 AP=0.814 P@K=0.801  recall@K: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.0)}


### Hyperparameters, seed stability, threshold

In [11]:
for nest in [100, 300, 600]:
    for ms in [256, 1024, 4096]:
        t0 = time.time(); m, s = fit_if(FULL, 0, nest, ms)
        top = np.argsort(-s)[:K]; print(f"n_estimators={nest:3d} max_samples={ms:4d} AP={average_precision_score(y, s):.4f} P@K={y[top].mean():.3f} time={time.time()-t0:.1f}s")
for nm, cols in [("FULL", FULL), ("NO_VALUE", NO_VALUE)]:
    runs = [fit_if(cols, sd)[1] for sd in range(5)]
    tops = [set(np.argsort(-s)[:K]) for s in runs]
    print(nm, "| spearman vs seed0:", np.round([spearmanr(runs[0], r)[0] for r in runs[1:]], 3), "| top-K Jaccard vs seed0:", np.round([len(tops[0] & t) / len(tops[0] | t) for t in tops[1:]], 3),
          "| AP per seed:", [round(average_precision_score(y, r), 3) for r in runs])
m, s_full = fit_if(FULL, 0, 300, 256)
print("sklearn default threshold (offset_=%.2f): flags %d rows (%.1f%%)" % (m.offset_, (s_full > 0.5).sum(), (s_full > 0.5).mean() * 100))
o = np.sort(s_full)[::-1]
for k in [1500, 2000, 2400, K, 2800, 3200, 4000]:
    sel = np.argsort(-s_full)[:k]; print(f"flag top {k:5d} ({k/len(s_full)*100:.2f}%) score>={o[k-1]:.4f} precision={y[sel].mean():.3f} recall={y[sel].sum()/K:.3f}")
h, b = np.histogram(s_full, bins=np.arange(0.35, 0.80, 0.025)); print("score histogram (bin start: count):", dict(zip(b.round(3)[:-1], h)))
print("score quantiles:", dict(zip([.5, .9, .95, .98, .99, .999], np.quantile(s_full, [.5, .9, .95, .98, .99, .999]).round(3))))
print("clean max score: %.3f | pseudo-anomaly p1 score: %.3f" % (s_full[~y].max(), np.quantile(s_full[y], .01)))

n_estimators=100 max_samples= 256 AP=0.8593 P@K=0.814 time=0.7s


n_estimators=100 max_samples=1024 AP=0.9124 P@K=0.830 time=0.8s


n_estimators=100 max_samples=4096 AP=0.9591 P@K=0.878 time=0.9s


n_estimators=300 max_samples= 256 AP=0.8707 P@K=0.816 time=1.8s


n_estimators=300 max_samples=1024 AP=0.9369 P@K=0.848 time=2.3s


...

n_estimators=600 max_samples=1024 AP=0.9224 P@K=0.836 time=4.9s


n_estimators=600 max_samples=4096 AP=0.9644 P@K=0.879 time=6.0s


FULL | spearman vs seed0: [0.98  0.975 0.982 0.982] | top-K Jaccard vs seed0: [0.915 0.901 0.916 0.898] | AP per seed: [0.871, 0.868, 0.868, 0.876, 0.871]


NO_VALUE | spearman vs seed0: [0.98  0.981 0.984 0.983] | top-K Jaccard vs seed0: [0.819 0.822 0.808 0.817] | AP per seed: [0.744, 0.731, 0.769, 0.715, 0.741]


sklearn default threshold (offset_=-0.50): flags 7155 rows (4.8%)
flag top  1500 (1.00%) score>=0.6694 precision=1.000 recall=0.557
flag top  2000 (1.33%) score>=0.6340 precision=1.000 recall=0.742
flag top  2400 (1.60%) score>=0.5579 precision=0.911 recall=0.811
flag top  2695 (1.80%) score>=0.5455 precision=0.816 recall=0.816
flag top  2800 (1.87%) score>=0.5424 precision=0.787 recall=0.818
flag top  3200 (2.13%) score>=0.5340 precision=0.695 recall=0.825
flag top  4000 (2.67%) score>=0.5229 precision=0.565 recall=0.838
score histogram (bin start: count): {np.float64(0.35): np.int64(8013), np.float64(0.375): np.int64(42284), np.float64(0.4): np.int64(41300), np.float64(0.425): np.int64(27130), np.float64(0.45): np.int64(16206), np.float64(0.475): np.int64(7912), np.float64(0.5): np.int64(3327), np.float64(0.525): np.int64(1254), np.float64(0.55): np.int64(362), np.float64(0.575): np.int64(66), np.float64(0.6): np.int64(82), np.float64(0.625): np.int64(234), np.float64(0.65): np.int64

### Non-circular check 1: ablation without order_value / residual
NO_VALUE has no access to order_value, so residual violations cannot leak in. Types it *can* see (A,B,C,F) vs the one it cannot (E):

In [12]:
_, s_nv = fit_if(NO_VALUE, 0)
for nm, s in [("FULL", s_full), ("NO_VALUE", s_nv)]:
    top = np.zeros(len(s), bool); top[np.argsort(-s)[:K]] = True
    print(f"{nm:9s} recall@K by type:", {t[0]: round(top[typ == t].mean(), 2) for t in TYPES}, "| false positives in top-K:", int((top & ~y).sum()))
print("rank-corr FULL vs NO_VALUE:", round(spearmanr(s_full, s_nv)[0], 3), "| top-K overlap:", round(len(set(np.argsort(-s_full)[:K]) & set(np.argsort(-s_nv)[:K])) / K, 3))

FULL      recall@K by type: {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.08)} | false positives in top-K: 495
NO_VALUE  recall@K by type: {'A': np.float64(1.0), 'B': np.float64(0.98), 'C': np.float64(1.0), 'E': np.float64(0.22), 'F': np.float64(0.11)} | false positives in top-K: 905
rank-corr FULL vs NO_VALUE: 0.967 | top-K overlap: 0.793


### Non-circular check 2: synthetic injection into a COPY of clean rows (model not retrained on them)
Only S6/S7 can be seen by the residual; S1,S2,S4,S5 are formula-consistent, S3 touches only the fee.

In [13]:
rs = np.random.RandomState(1)
clean_idx = np.where(~y & ~dup)[0]; pick = rs.choice(clean_idx, 5000, replace=False)
base = raw.iloc[pick].drop(columns="row_id").reset_index(drop=True)
def consistent(d):
    s_ = d.quantity * d.unit_price * (1 - d.discount_pct.fillna(0) / 100); d["order_value"] = (s_ + d.shipping_cost.fillna(0) + d.tax_amount).round(2); return d
nn = 600
parts = {
 "S1 quantity x10, fee/tax scaled, formula-consistent": consistent(base.iloc[:nn].assign(quantity=lambda d: d.quantity * 10, tax_amount=lambda d: d.tax_amount * 10, platform_fee=lambda d: d.platform_fee * 10)),
 "S2 unit_price x6, fee/tax scaled, formula-consistent": consistent(base.iloc[nn:2*nn].assign(unit_price=lambda d: d.unit_price * 6, tax_amount=lambda d: d.tax_amount * 6, platform_fee=lambda d: d.platform_fee * 6)),
 "S3 platform_fee x8 only": base.iloc[2*nn:3*nn].assign(platform_fee=lambda d: d.platform_fee * 8),
 "S4 discount 70%, fee/tax kept, formula-consistent": consistent(base.iloc[3*nn:4*nn].assign(discount_pct=70.0)),
 "S5 shipping 150, formula-consistent": consistent(base.iloc[4*nn:5*nn].assign(shipping_cost=150.0)),
 "S6 account_age 1-5d, nothing else": base.iloc[5*nn:6*nn].assign(account_age_days=lambda d: rs.randint(1, 6, len(d))),
 "S7 order_value x15 only": base.iloc[6*nn:7*nn].assign(order_value=lambda d: (d.order_value * 15).round(2)),
}
ctrl = base.iloc[7*nn:]
thr = np.quantile(s_full[~dup], 1 - 0.018)
print("threshold = 98.2th percentile of training scores:", round(thr, 4), "| control FPR:", round((-m.score_samples(fe.transform(ctrl)) > thr).mean(), 4), f"(n={len(ctrl)})")
for k, dd in parts.items():
    s = -m.score_samples(fe.transform(dd)); print(f"{k:55s} recall={np.mean(s > thr):.2f}  median score={np.median(s):.3f}")
m_nv, _ = fit_if(NO_VALUE, 0); thr_nv = np.quantile(-m_nv.score_samples(X.loc[~dup, NO_VALUE]), 1 - 0.018)
print("-- same test, NO_VALUE model --")
for k, dd in parts.items():
    s = -m_nv.score_samples(fe.transform(dd)[NO_VALUE]); print(f"{k:55s} recall={np.mean(s > thr_nv):.2f}")

threshold = 98.2th percentile of training scores: 0.5452 | control FPR: 0.0012 (n=800)
S1 quantity x10, fee/tax scaled, formula-consistent     recall=0.06  median score=0.500
S2 unit_price x6, fee/tax scaled, formula-consistent    recall=0.05  median score=0.472


S3 platform_fee x8 only                                 recall=0.07  median score=0.494
S4 discount 70%, fee/tax kept, formula-consistent       recall=0.52  median score=0.546
S5 shipping 150, formula-consistent                     recall=0.36  median score=0.532
S6 account_age 1-5d, nothing else                       recall=0.08  median score=0.489


S7 order_value x15 only                                 recall=0.85  median score=0.565


-- same test, NO_VALUE model --
S1 quantity x10, fee/tax scaled, formula-consistent     recall=0.09
S2 unit_price x6, fee/tax scaled, formula-consistent    recall=0.06
S3 platform_fee x8 only                                 recall=0.12
S4 discount 70%, fee/tax kept, formula-consistent       recall=0.81
S5 shipping 150, formula-consistent                     recall=0.40
S6 account_age 1-5d, nothing else                       recall=0.23


S7 order_value x15 only                                 recall=0.01


### Comparison models (robust-scaled)

In [14]:
Zs = RobustScaler().fit(X.loc[~dup, FULL]); Z = Zs.transform(X[FULL])
print("RobustScaler IQR per feature:", dict(zip(FULL, Zs.scale_.round(3))))
idx = np.random.RandomState(0).choice(np.where(~dup)[0], 30000, replace=False)
t0 = time.time(); ee = EllipticEnvelope(contamination=0.018, support_fraction=0.9, random_state=0).fit(Z[idx]); s_ee = -ee.score_samples(Z)
print(f"EllipticEnvelope(robust-scaled FULL, fit on 30k) {time.time()-t0:.0f}s AUC={roc_auc_score(y, s_ee):.3f} AP={average_precision_score(y, s_ee):.3f} top-K overlap with IF={len(set(np.argsort(-s_ee)[:K]) & set(np.argsort(-s_full)[:K]))/K:.2f}")
for kk in [35, 300]:
    t0 = time.time(); lof = LocalOutlierFactor(n_neighbors=kk, novelty=True, n_jobs=-1).fit(Z[idx]); s_l = -lof.score_samples(Z)
    print(f"LOF k={kk:3d} (30k train sample) {time.time()-t0:.0f}s AUC={roc_auc_score(y, s_l):.3f} AP={average_precision_score(y, s_l):.3f}")
print("anomalies per type in a 30k sample:", {t[0]: int((typ[idx] == t).sum()) for t in TYPES}, "-> an LOF neighbourhood smaller than that cluster is 'masked'")

RobustScaler IQR per feature: {'log_quantity': np.float64(0.405), 'log_unit_price': np.float64(1.223), 'discount_pct': np.float64(15.0), 'log_shipping': np.float64(0.748), 'log1p_tax_rate': np.float64(0.092), 'log_fee_rate': np.float64(0.399), 'price_vs_product': np.float64(0.194), 'log_order_value': np.float64(1.323), 'log_value_ratio': np.float64(1.0), 'log_acct_age': np.float64(1.107), 'log_ship_ratio': np.float64(0.24), 'log_fee_to_value': np.float64(0.464)}


EllipticEnvelope(robust-scaled FULL, fit on 30k) 5s AUC=0.991 AP=0.860 top-K overlap with IF=0.83


LOF k= 35 (30k train sample) 18s AUC=0.434 AP=0.098


LOF k=300 (30k train sample) 43s AUC=1.000 AP=0.998
anomalies per type in a 30k sample: {'A': 109, 'B': 106, 'C': 102, 'E': 94, 'F': 106} -> an LOF neighbourhood smaller than that cluster is 'masked'


## 6. Save / load round trip (reproducibility test of the pipeline design)

In [15]:
pipe = Pipeline([("features", OrderFeatures()), ("iforest", IsolationForest(n_estimators=300, max_samples=256, random_state=42, n_jobs=-1))])
pipe.fit(train_raw)
scores_train = -pipe.score_samples(raw)
bundle = {"pipeline": pipe, "feature_order": FULL, "threshold": float(np.quantile(-pipe.score_samples(train_raw), 1 - 0.018)),
          "contamination": 0.018, "sklearn": __import__("sklearn").__version__}
buf = io.BytesIO(); joblib.dump(bundle, buf); print("bundle size KB:", round(len(buf.getvalue()) / 1024), "| lookup entries:", len(pipe[0].prod_med_price_), "products")
buf.seek(0); b2 = joblib.load(buf)
sc2 = -b2["pipeline"].score_samples(raw)
print("identical scores after reload:", np.allclose(scores_train, sc2), "| threshold:", round(bundle["threshold"], 4))
one = raw.iloc[[12345]]; print("single-row score equals batch score:", np.isclose(-b2["pipeline"].score_samples(one)[0], sc2[12345]))
unseen = one.copy(); unseen["product_id"] = "P99999"; unseen["category"] = "automotive "; print("unseen product + dirty label scores without error:", float(-b2["pipeline"].score_samples(unseen)[0]))
flag = scores_train > bundle["threshold"]; print("flagged:", flag.sum(), f"({flag.mean()*100:.2f}%) precision vs pseudo-labels {y[flag].mean():.3f} recall {y[flag].sum()/y.sum():.3f}")

bundle size KB: 2897 | lookup entries: 1200 products


identical scores after reload: True | threshold: 0.5573
single-row score equals batch score: True
unseen product + dirty label scores without error: 0.5303547623211465
flagged: 2702 (1.80%) precision vs pseudo-labels 0.814 recall 0.816


## 7. Explaining flagged orders: robust per-feature deviation (and SHAP check)

In [16]:
med = X.loc[~dup].median(); iqr = (X.loc[~dup].quantile(.75) - X.loc[~dup].quantile(.25)); scale = np.maximum(iqr / 1.349, 0.01)
Zr = (X - med) / scale
res_tbl = pd.DataFrame({"row_id": raw.row_id, "order_id": raw.order_id, "score": scores_train, "pseudo_type": typ})
ex = {}
for t in TYPES:
    i = res_tbl[res_tbl.pseudo_type == t].score.idxmax() if t != "A_bulk_highprice" else res_tbl[res_tbl.pseudo_type == t].score.sort_values().index[len(res_tbl[res_tbl.pseudo_type == t]) // 2]
    ex[t] = i
for t, i in ex.items():
    top3 = Zr.loc[i].abs().sort_values(ascending=False).head(3).index
    r = raw.loc[i]
    print(f"\n[{t}] row_id={i} order_id={r.order_id} score={scores_train[i]:.3f} | qty={r.quantity} price={r.unit_price} (product median {pm[i]:.2f}) disc={r.discount_pct} ship={r.shipping_cost} tax={r.tax_amount} fee={r.platform_fee} value={r.order_value} (formula expects {exp_[i]:.2f}) acct_age={r.account_age_days}d")
    print("   top deviations (robust z):", {f: round(float(Zr.loc[i, f]), 1) for f in top3})
try:
    import shap
    expl = shap.TreeExplainer(pipe[1]); sv = expl.shap_values(X.iloc[list(ex.values())][FULL])
    print("\nSHAP (TreeExplainer on IsolationForest) works; top feature per example:", {t[0]: FULL[int(np.argmin(sv[j]))] for j, t in enumerate(ex)})
    print("(negative SHAP = pushes toward 'more anomalous' for IsolationForest.score_samples)")
except Exception as e:
    print("SHAP failed:", type(e).__name__, str(e)[:100])


[A_bulk_highprice] row_id=130921 order_id=ORD0041287 score=0.718 | qty=33 price=1621.25 (product median 40.17) disc=0.0 ship=4.05 tax=3.25 fee=3.81 value=59736.08 (formula expects 53508.55) acct_age=211d
   top deviations (robust z): {'price_vs_product': 25.7, 'log_fee_rate': -23.8, 'log_fee_to_value': -21.0}

[B_extreme_discount] row_id=34251 order_id=ORD0039045 score=0.762 | qty=1 price=5.77 (product median 7.24) disc=94.0 ship=2.92 tax=0.48 fee=0.37 value=3352.8 (formula expects 3.75) acct_age=2339d
   top deviations (robust z): {'log_value_ratio': 679.7, 'log_fee_to_value': -19.4, 'log_ship_ratio': 11.9}

[C_ship_or_lowprice] row_id=110845 order_id=ORD0004365 score=0.747 | qty=10 price=1.96 (product median 167.66) disc=40.0 ship=350.26 tax=133.51 fee=103.24 value=1192.36 (formula expects 495.53) acct_age=2863d
   top deviations (robust z): {'log_value_ratio': 87.8, 'log1p_tax_rate': 35.5, 'price_vs_product': -30.9}

[E_newacct_value_inflated] row_id=77885 order_id=ORD0043376 score


SHAP (TreeExplainer on IsolationForest) works; top feature per example: {'A': 'log_fee_to_value', 'B': 'log_fee_to_value', 'C': 'log_fee_rate', 'E': 'log_value_ratio', 'F': 'log_acct_age'}
(negative SHAP = pushes toward 'more anomalous' for IsolationForest.score_samples)


## 8. Follow-up: why fee-inflated rows (F) and single-feature anomalies are the weak spot; remedies
Larger max_samples, dropping the noisy ship_ratio, and a max-|robust-z| rule layer.

In [17]:
fF=(typ=="F_fee_inflated")
def full_report(name,s,k=K):
    top=np.zeros(len(s),bool); top[np.argsort(-s)[:k]]=True
    print(f"{name:50s} AP={average_precision_score(y,s):.3f} P@K={top[y].mean():.3f} recall@K",{t[0]:round(top[typ==t].mean(),2) for t in TYPES},"FP",int((top&~y).sum()))
lean=["log_quantity","log_unit_price","discount_pct","log_shipping","log1p_tax_rate","log_fee_rate","price_vs_product","log_value_ratio","log_acct_age"]
for nm,cols,ms in [("FULL ms256",FULL,256),("FULL ms4096",FULL,4096),("LEAN9 (no order_value, ship_ratio, fee_to_value) ms256",lean,256),("LEAN9 ms4096",lean,4096),
                   ("FULL minus ship_ratio ms4096",[c for c in FULL if c!="log_ship_ratio"],4096)]:
    full_report(nm,fit_if(cols,0,300,ms,Xd=None)[1])
# max |robust z| rule
med=X.loc[~dup].median(); iqr=X.loc[~dup].quantile(.75)-X.loc[~dup].quantile(.25); sc=np.maximum(iqr/1.349,0.01)
Zr=((X-med)/sc).abs(); mz=Zr.max(axis=1).values
full_report("max |robust z| over FULL (rule baseline)",mz)
for t_ in [5,6,8,10]: print("max|z|>",t_,"flags",(mz>t_).sum(),"recall",round(((mz>t_)&y).sum()/y.sum(),3),"precision",round(y[mz>t_].mean(),3),"F recall",round((mz>t_)[fF].mean(),3))
print("clean max|z| quantiles",np.quantile(mz[~y],[.5,.99,.999,1]).round(1),"F max|z| q",np.quantile(mz[fF],[0,.05,.5]).round(1))
# IF + synthetic with ms4096 and EE on synthetic
Zs=RobustScaler().fit(X.loc[~dup,FULL]); 
idx=np.random.RandomState(0).choice(np.where(~dup)[0],30000,replace=False)
ee=EllipticEnvelope(contamination=0.018,support_fraction=0.9,random_state=0).fit(Zs.transform(X[FULL])[idx]); thr_ee=np.quantile(-ee.score_samples(Zs.transform(X.loc[~dup,FULL])),1-0.018)
m4,s4=fit_if(FULL,0,300,4096); thr4=np.quantile(s4[~dup],1-0.018)
rs=np.random.RandomState(1); base=raw.iloc[rs.choice(np.where(~y&~dup)[0],4800,replace=False)].drop(columns="row_id").reset_index(drop=True)
def consistent(d):
    s_=d.quantity*d.unit_price*(1-d.discount_pct.fillna(0)/100); d["order_value"]=(s_+d.shipping_cost.fillna(0)+d.tax_amount).round(2); return d
nn=600
parts={"S1 qty x10":consistent(base.iloc[:nn].assign(quantity=lambda d:d.quantity*10,tax_amount=lambda d:d.tax_amount*10,platform_fee=lambda d:d.platform_fee*10)),
"S2 price x6":consistent(base.iloc[nn:2*nn].assign(unit_price=lambda d:d.unit_price*6,tax_amount=lambda d:d.tax_amount*6,platform_fee=lambda d:d.platform_fee*6)),
"S3 fee x8":base.iloc[2*nn:3*nn].assign(platform_fee=lambda d:d.platform_fee*8),
"S4 disc 70%":consistent(base.iloc[3*nn:4*nn].assign(discount_pct=70.0)),
"S5 ship 150":consistent(base.iloc[4*nn:5*nn].assign(shipping_cost=150.0)),
"S6 acct 1-5d":base.iloc[5*nn:6*nn].assign(account_age_days=lambda d:rs.randint(1,6,len(d))),
"S7 value x15":base.iloc[6*nn:7*nn].assign(order_value=lambda d:(d.order_value*15).round(2))}
ctrl=base.iloc[7*nn:]
print("%-14s %8s %8s %8s %8s"%("","IF256","IF4096","EE","maxz>6"))
def sc3(d):
    F=fe.transform(d); return (-m.score_samples(F)>thr_q256), (-m4.score_samples(F)>thr4), (-ee.score_samples(Zs.transform(F[FULL]))>thr_ee), ((((F[FULL]-med)/sc).abs().max(axis=1))>6)
thr_q256=np.quantile(s_full[~dup],1-0.018) if 's_full' in dir() else None
m,s_256=fit_if(FULL,0,300,256); thr_q256=np.quantile(s_256[~dup],1-0.018)
for k,d in [("control",ctrl)]+list(parts.items()):
    r=sc3(d); print("%-14s"%k," ".join("%8.2f"%np.mean(x) for x in r))
print("on the real pseudo-labels at same thresholds: F recall IF256/IF4096/EE/maxz:", round((s_256>thr_q256)[fF].mean(),2), round((s4>thr4)[fF].mean(),2), round((-ee.score_samples(Zs.transform(X[FULL]))>thr_ee)[fF].mean(),2), round((mz>6)[fF].mean(),2))
for nm,s,th in [("IF256",s_256,thr_q256),("IF4096",s4,thr4)]:
    fl=s>th; print(nm,"flagged",fl.sum(),"precision",round(y[fl].mean(),3),"recall",round(y[fl].sum()/y.sum(),3), {t[0]:round(fl[typ==t].mean(),2) for t in TYPES})

FULL ms256                                         AP=0.871 P@K=0.816 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.08)} FP 495


FULL ms4096                                        AP=0.956 P@K=0.874 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.37)} FP 339


LEAN9 (no order_value, ship_ratio, fee_to_value) ms256 AP=0.861 P@K=0.814 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(0.99), 'E': np.float64(1.0), 'F': np.float64(0.08)} FP 500


LEAN9 ms4096                                       AP=0.916 P@K=0.836 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.18)} FP 443


FULL minus ship_ratio ms4096                       AP=0.977 P@K=0.917 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.58)} FP 225
max |robust z| over FULL (rule baseline)           AP=0.922 P@K=0.824 recall@K {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.12)} FP 474
max|z|> 5 flags 10959 recall 0.981 precision 0.241 F recall 0.903
max|z|> 6 flags 4365 recall 0.935 precision 0.578 F recall 0.676
max|z|> 8 flags 2631 recall 0.82 precision 0.84 F recall 0.099
max|z|> 10 flags 2234 recall 0.803 precision 0.969 F recall 0.011
clean max|z| quantiles [ 2.3  6.4  9.1 17. ] F max|z| q [4.1 4.8 6.5]


                  IF256   IF4096       EE   maxz>6


control            0.00     0.00     0.00     0.01
S1 qty x10         0.06     0.17     0.04     0.46
S2 price x6        0.05     0.12     0.00     1.00
S3 fee x8          0.07     0.50     0.00     0.84


S4 disc 70%        0.52     0.81     0.08     1.00
S5 ship 150        0.36     0.67     0.33     0.65
S6 acct 1-5d       0.08     0.16     0.00     1.00
S7 value x15       0.85     0.99     1.00     1.00
on the real pseudo-labels at same thresholds: F recall IF256/IF4096/EE/maxz: 0.08 0.37 0.0 0.68
IF256 flagged 2702 precision 0.815 recall 0.817 {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.08)}


IF4096 flagged 2702 precision 0.873 recall 0.875 {'A': np.float64(1.0), 'B': np.float64(1.0), 'C': np.float64(1.0), 'E': np.float64(1.0), 'F': np.float64(0.37)}
